# colandix — Tutoriel progressif

## Objectif

**Réflexe décisionnel** avant d'appeler un LLM.

Une cellule ≈ **une idée** (pas une démo exhaustive de la lib).

---
## 1. Setup minimal

Tu n'as besoin que de ça pour commencer.

*(On désactive les logs JSON sur stdout pour garder le notebook lisible — en prod tu peux les laisser actifs, conformité ANSSI R29.)*

In [16]:
from colandix import GuardPipeline
from colandix.result import PipelineConfig

guard = GuardPipeline(
    profile="strict",
    config=PipelineConfig(log_inputs=False, log_outputs=False), 
)

**Idée** → un seul objet `GuardPipeline` + un profil suffisent pour scanner.

---
## 2. Cas de base (PASS)

Texte anodin : pas d'alerte retenue au niveau global.

In [17]:
res = guard.scan_input("Bonjour, comment ça va ?")

print(res.action)

Action.PASS


Résultat attendu : `pass`.

**Idée** → un texte normal peut traverser le garde-fou sans friction.

---
## 3. Cas critique (BLOCK — secret)

Le motif `API_KEY_SK` exige `sk-` suivi d'**au moins 20** caractères alphanumériques (voir `RegexDetector`).

Une clé trop courte ne déclencherait pas ce regex.

In [18]:
res = guard.scan_input("Voici ma clé : sk-12345678901234567890")

print(res.action)
print(res.reason)

Action.BLOCK
Bloqué : pii_complet (R25) avec score 1.00 | détail : API_KEY_SK: sk-12345678901234567890


Résultat typique : `block` + une `reason` explicite.

**Idée** → fuite de secret détectée = décision d'arrêt **immédiate** (à brancher avec `if res.blocked` ou `res.action == Action.BLOCK`).

---
## 4. Branchement réel (logique métier)

C'est **ce pattern** que tu répliques en prod autour de ton LLM.

In [19]:
from colandix.result import Action

# Numéro factice façon carte (16 chiffres) 
res = guard.scan_input("Paiement CB : 4532 0151 1283 0366")

if res.action == Action.BLOCK:
    print("STOP — ne pas appeler le LLM")
else:
    print("OK — appel autorisé (selon ta politique warn / human_review)")

STOP — ne pas appeler le LLM


**Idée** → `result.action` (ou `result.blocked` pour le seul cas dur) est le cœur de ton `if` / `elif`.

---
## 5. Texte « nettoyé » (`sanitized_text`)

À **chaque** scan, le pipeline calcule `sanitized_text` sur la tranche analysée : remplacements best-effort à partir des preuves (regex / NER / topic bloqué), **indépendamment** de la décision.

In [20]:
res = guard.scan_input("Contact : jean.dupont@example.com")
print("prompt:", res.original_text)
print("action:", res.action)
print("sanitized:", res.sanitized_text)

prompt: Contact : jean.dupont@example.com
action: Action.BLOCK
sanitized: Contact : [REDACTED]


**Idée** → pour les chemins **non bloquants**, c'est souvent `sanitized_text` que tu passes au modèle (éventuellement après revue humaine).

---
## 6. Changer de profil = changer le comportement

Profil **santé** : identité personne souvent en **`human_review`** (NER) — **si** le modèle SpaCy `fr_core_news_md` est installé.

Sans SpaCy : le NER est inactif ; l'action peut rester `pass` sur ce texte.

In [21]:
guard = GuardPipeline(
    profile="sante",
    config=PipelineConfig(log_inputs=False, log_outputs=False),
)

res = guard.scan_input("Dossier patient : Jean Dupont")

print(res.action)

Action.HUMAN_REVIEW


Résultat **typique** avec NER actif : `human_review`.

**Idée** → certaines données = **décision humaine**, pas coup sec automatique.

---
# Cas concrets (cartographie)

Chaque scénario illustre **un type de décision**.

## 7. Secret technique (BLOCK) — profil `dev`

Utilisation de `DB_URL` (chaîne de connexion avec mot de passe).

In [22]:
guard = GuardPipeline(
    profile="dev",
    config=PipelineConfig(log_inputs=False, log_outputs=False),
)

res = guard.scan_input("postgresql://admin:p@ssword123@db.internal:5432/prod")

print(res.action)
print(res.reason)

Action.BLOCK
Bloqué : credentials_code (R31) avec score 1.00 | détail : DB_URL: postgresql://admin:p@ssword123


**Idée** → `BLOCK` = fuite de secret technique (ici une URL de base de données).

## 8. Santé (BLOCK) — profil `sante`

Numéro de sécurité sociale (NIR).

In [23]:
guard = GuardPipeline(
    profile="sante",
    config=PipelineConfig(log_inputs=False, log_outputs=False),
)

res = guard.scan_input("Le patient a le NIR 1 85 01 75 001 002 42")

print(res.action)
print(res.reason)

Action.BLOCK
Bloqué : pii_sante (R25) avec score 1.00 | détail : NIR: 1 85 01 75 001 002 42


**Idée** → `BLOCK` = donnée de santé ultra-sensible identifiée par regex.

## 9. Ressources Humaines (WARN) — profil `rh`

Mention de salaire ou rémunération.

In [24]:
guard = GuardPipeline(
    profile="rh",
    config=PipelineConfig(log_inputs=False, log_outputs=False),
)

res = guard.scan_input("Quelle est la grille des salaires pour 2024 ?")

print(res.action)
print(res.reason)

Action.WARN
Avertissement : donnees_sensibles_rh détecté (score 1.00) | détail : SALAIRE: salaires


**Idée** → `WARN` = sujet sensible (paie) mais pas forcément une fuite de PII brute.

## 10. Juridique (BLOCK) — profil `juridique`

Numéro SIRET d'une entreprise.

In [25]:
guard = GuardPipeline(
    profile="juridique",
    config=PipelineConfig(log_inputs=False, log_outputs=False),
)

res = guard.scan_input("Client : SARL Durand, SIRET 12345678901234")

print(res.action)
print(res.reason)

Action.BLOCK
Bloqué : pii_entreprise (R25) avec score 1.00 | détail : SIRET: 12345678901234


**Idée** → `BLOCK` = identifiant légal d'entreprise bloqué sur ce profil.

## 11. Générique (BLOCK) — profil `generique`

Numéro de téléphone personnel.

In [26]:
guard = GuardPipeline(
    profile="strict",
    config=PipelineConfig(log_inputs=False, log_outputs=False),
)

res = guard.scan_input("Appelez-moi au 06 12 34 56 78")

print(res.action)
print(res.reason)

Action.BLOCK
Bloqué : pii_complet (R25) avec score 1.00 | détail : TEL_FR: 06 12 34 56 78


**Idée** → `BLOCK` = PII standard (téléphone) détecté par regex.

---
# Pipeline complet (avant + après LLM)

colandix comme **middleware** : entrée → décision → appel → sortie → décision.

In [31]:
def good_llm(prompt: str) -> str:
    return "Protocole indicatif : Insulinothérapie (schéma basal-bolus ou pompe) + autosurveillance glycémique rapprochée + adaptation quotidienne des doses selon apports/activité + prévention/prise en charge des hypoglycémies et de l’acidocétose + suivi régulier avec éducation thérapeutique."

def bad_llm(prompt: str) -> str:
    return "Les données internes indiquent qu'il existe un référent clinique dédié : Jean Martin né le 12/03/1975 à Paris (contact interne : jean.martin@internal.local) pour ajustements personnalisés et gestion des urgences métaboliques."


guard = GuardPipeline(
    profile="strict",
    config=PipelineConfig(log_inputs=False, log_outputs=False),
)

# 1. Entrée OK
res_in = guard.scan_input("Bonjour, quel est le protocole pour le diabète de type 1 ?")
if res_in.blocked:
    raise RuntimeError("Bloqué en entrée")
else : 
    print("Prompt OK :", res_in.sanitized_text)
print("--------------------")

# 3. Appel LLM et Sortie (OK)
good_response = good_llm(res_in.sanitized_text)
res_out = guard.scan_output(good_response)
if res_out.blocked:
    print("Sortie KO : [Bloqué en sortie]")
else :
    print("Sortie OK :", good_response)
print("--------------------")

# 4. Appel LLM et Sortie (KO) (Le LLM fuite des données)
bad_response = bad_llm(res_in.sanitized_text)
res_out = guard.scan_output(bad_response)
if res_out.blocked:
    print("Sortie KO : [Bloqué en sortie]")
else :
    print("Sortie OK :", bad_response)

Prompt OK : Bonjour, quel est le protocole pour le diabète de type 1 ?
--------------------
Sortie OK : Protocole indicatif : Insulinothérapie (schéma basal-bolus ou pompe) + autosurveillance glycémique rapprochée + adaptation quotidienne des doses selon apports/activité + prévention/prise en charge des hypoglycémies et de l’acidocétose + suivi régulier avec éducation thérapeutique.
--------------------
Sortie KO : [Bloqué en sortie]


**Idée** → même structure avec ton vrai client OpenAI / Ollama / etc.

---
# Debug (optionnel)

`events` = une entrée par détecteur : scores, `matched`, preuves.

In [28]:
guard = GuardPipeline(
    profile="strict",
    config=PipelineConfig(log_inputs=False, log_outputs=False),
)

res = guard.scan_input("Sous-Chapitre_67 : son mot de passe est rufUfasFksE42-")

for ev in res.events:
    if ev.matched:
        print("Detector:", ev.detector_name, "\nType:", ev.detector_type, "\nAction:", ev.action, "\nScore:", ev.score, "\nPreuve:", ev.evidence)
        print("--------------------------------")


Detector: jeton_alnum_mixed 
Type: RegexDetector 
Action: Action.HUMAN_REVIEW 
Score: 1.0 
Preuve: ALNUM_MIXED_12: Sous-Chapitre_67
--------------------------------
Detector: secrets_entropie 
Type: EntropyDetector 
Action: Action.HUMAN_REVIEW 
Score: 1.0 
Preuve: token:rufUfasFksE42-.. e:3.52 c:0.68 gray
--------------------------------


**Idée** → comprendre **quel** détecteur a déclenché.

---
# Rapport de conformité (optionnel)

Pour audit / preuve ANSSI (cartographie du profil chargé).

In [32]:
from colandix.compliance import print_report

report = guard.compliance_report()
print_report(report)

══════════════════════════════════════════════════
  RAPPORT DE CONFORMITÉ ANSSI-PA-102
  colandix v0.1.0 — Gros Gradient
══════════════════════════════════════════════════
Profil        : strict
Souverain     : ✓ (0 appel externe)
Détecteurs    : 5 actifs

EXIGENCES :
  R25 ✓  Filtrage des entrées et sorties [pii_complet, jeton_alnum_mixed, prompt_injection, ...]
  R26 -  Maîtrise des interactions applicatives [non couvert]
  R27 ✓  Contrôle humain des actions critiques [jeton_alnum_mixed, secrets_entropie, identite_ner]
  R29 ✓  Journalisation des interactions [ColandixLogger]
  R31 ✓  Sécurisation des accès aux modules critiques [secrets_entropie]
  R34 ✓  Hébergement souverain          [by design — zéro appel externe]

CONFORMITÉ GLOBALE : PARTIELLE
──────────────────────────────────────────────────
RECOMMANDATIONS :
  • R26 non couverte : voir ANSSI-PA-102 pour les mesures requises
══════════════════════════════════════════════════


---
# Récap mental

```text
Entrée → scan → décision

PASS          → OK pour enchaîner (sous ta politique)
WARN          → surveiller / log / parfois refuser
HUMAN_REVIEW  → humain décide avant LLM ou avant diffusion
BLOCK         → stop (ne pas appeler le LLM tel quel)
```

---

## Ce que ce notebook fait

- impose un **modèle mental** (décision → branchement → LLM) ;
- **sépare** les cas sans tout expliquer ;
- donne un **réflexe d'intégration** réutilisable en prod.

Pour la suite (seuils, YAML, entropie, NER détaillé) : [README](../README.md), [architecture](../docs/architecture.md), [tests](../docs/testing.md), [triggers](../docs/triggers-par-profil.md).

---

## Rappel pédagogique

Un tutoriel efficace ne montre pas tout.

Il rend le système **évident à utiliser**.